1.Install and Import Dependencies (pip install, import)
2.Configure Parameters (based on the original Config.py)
3.Chinese Word Segmentation + Custom Dataset (modified DataSet.py)
4.Transformer Model Definition (modified Transformer.py)
5.Training Function train_model (modified train.py)
6.Testing Function test_model
7.Main Pipeline Execution (load data → build model → train → test)

In [ ]:
# ========= preprocess =========
from collections import Counter
import numpy as np
import jieba
import re
import pandas as pd
import os
import torch
import random

def set_seed(seed=142):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)    

# file path
data_path = '/kaggle/input/douban-mgh'
train_file = 'train_pinggu.csv'
valid_file = 'test_pinggu.csv'
test_file = 'test_pinggu.csv'

train_df = pd.read_csv(os.path.join(data_path, train_file)) 

# Stop vocabulary loading
with open('/kaggle/input/cn-stopwords/cn_stopwords.txt', encoding='utf-8') as f:
    stop_words = set(line.strip() for line in f if line.strip())

# word frequency statistic
def get_word_freq(texts):
    all_tokens = []
    for text in texts:
        x = re.sub(r"[^\u4e00-\u9fa5]", "", str(text))
        tokens = jieba.lcut(x)
        all_tokens.extend(tokens)
    return Counter(all_tokens)

word_freq = get_word_freq(train_df['text'].tolist()) 



In [ ]:
# ========= Configuration parameters=========

# Hyperparameters
fix_length = 64
batch_size = 64  
label_list = ['negative', 'positive']
class_number = len(label_list)
epochs = 60
#learning_rate = 3e-4
#learning_rate =2e-4
learning_rate =1e-3

# ========= jieba_tokenize+ Customize Dataset =========
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import jieba
import re


def jieba_tokenize(x, word_freq=None):
    x = re.sub('[^\u4e00-\u9fa5]', "", str(x))
    tokens = jieba.lcut(x)
    
    if word_freq:
        tokens = [
            t for t in tokens 
            if word_freq[t] < 10000  # Custom frequency range
            and t not in stop_words
            and len(t) > 1
        ]
    else:
        tokens = [t for t in tokens if t not in stop_words and len(t) > 1]
    
    return tokens

from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-chinese")

class CommentDataset(Dataset):
    def __init__(self, texts, labels, word_freq=None, max_len=256):
        #jieba_tokenize + cleaning
        texts = [" ".join(jieba_tokenize(t, word_freq)) for t in texts]
        self.encodings = tokenizer(
            texts,
            padding='max_length',
            truncation=True,
            max_length=max_len,
            return_tensors='pt'
        )
        self.labels = torch.tensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

# ========= Transformer  =========
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import copy

class Transformer(nn.Module):
    def __init__(self, vocab_size=21128, seq_len=fix_length, n_class=class_number,
                 embed_dim=256, dim_model=256, dropout=0.2, num_head=8, hidden=512, num_encoder=4):
        super(Transformer, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.position_embedding = PositionalEncoding(embed_dim, seq_len, dropout)
        self.encoder = Encoder(dim_model, num_head, hidden, dropout)
        self.encoders = nn.ModuleList([copy.deepcopy(self.encoder) for _ in range(num_encoder)])
        self.fc = nn.Linear(seq_len * dim_model, n_class)

    def forward(self, input_ids, attention_mask=None):
        out = self.embedding(input_ids)
        out = self.position_embedding(out)
        for encoder in self.encoders:
            out = encoder(out)
        out = out.view(out.size(0), -1)
        return self.fc(out)

class PositionalEncoding(nn.Module):
    def __init__(self, embed, seq_len, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(seq_len, embed)
        position = torch.arange(0, seq_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, embed, 2) * -(np.log(10000.0) / embed))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.pe = pe.unsqueeze(0)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :].to(x.device)
        return self.dropout(x)

class Encoder(nn.Module):
    def __init__(self, dim_model, num_head, hidden, dropout):
        super(Encoder, self).__init__()
        self.attn = MultiHeadAttention(dim_model, num_head, dropout)
        self.ff = PositionWiseFeedForward(dim_model, hidden, dropout)

    def forward(self, x):
        x = self.attn(x)
        x = self.ff(x)
        return x

class MultiHeadAttention(nn.Module):
    def __init__(self, dim_model, num_head, dropout=0.1):
        super().__init__()
        assert dim_model % num_head == 0
        self.dim_head = dim_model // num_head
        self.num_head = num_head
        self.q_linear = nn.Linear(dim_model, dim_model)
        self.k_linear = nn.Linear(dim_model, dim_model)
        self.v_linear = nn.Linear(dim_model, dim_model)
        self.fc = nn.Linear(dim_model, dim_model)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(dim_model)

    def forward(self, x):
        batch_size, seq_len, dim = x.size()
        Q = self.q_linear(x).view(batch_size, seq_len, self.num_head, self.dim_head).transpose(1, 2)
        K = self.k_linear(x).view(batch_size, seq_len, self.num_head, self.dim_head).transpose(1, 2)
        V = self.v_linear(x).view(batch_size, seq_len, self.num_head, self.dim_head).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(self.dim_head)
        attn = F.softmax(scores, dim=-1)
        context = torch.matmul(attn, V)
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, dim)
        out = self.fc(context)
        out = self.dropout(out)
        return self.norm(out + x)

class PositionWiseFeedForward(nn.Module):
    def __init__(self, dim_model, hidden, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(dim_model, hidden)
        self.fc2 = nn.Linear(hidden, dim_model)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(dim_model)

    def forward(self, x):
        out = self.fc2(F.relu(self.fc1(x)))
        out = self.dropout(out)
        return self.norm(out + x)

# ========= TextCNN =========
class TextCNN(nn.Module):
    def __init__(self, vocab_size=21128, embed_dim=128, class_num=2, kernel_nums=128, kernel_sizes=[3,4,5], dropout=0.7):
        super(TextCNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.convs = nn.ModuleList([
            nn.Conv2d(1, kernel_nums, (k, embed_dim)) for k in kernel_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(len(kernel_sizes) * kernel_nums, class_num)

    def forward(self, input_ids):
        # input_ids: [batch_size, seq_len]
        x = self.embedding(input_ids)  # [batch_size, seq_len, embed_dim]
        x = x.unsqueeze(1)             # [batch_size, 1, seq_len, embed_dim]
        x = [F.relu(conv(x)).squeeze(3) for conv in self.convs]  # conv: [batch_size, kernel_nums, ~]
        x = [F.max_pool1d(i, i.size(2)).squeeze(2) for i in x]   # max pool: [batch_size, kernel_nums]
        x = torch.cat(x, 1)           # [batch_size, kernel_nums * len(kernel_sizes)]
        x = self.dropout(x)
        out = self.fc(x)              # [batch_size, class_num]
        return out


In [ ]:
# ========= train and test function =========
import torch.nn.functional as F
from sklearn.metrics import classification_report, accuracy_score
from tqdm import tqdm
import matplotlib.pyplot as plt

def train_model(model, train_loader, val_loader, device, name='Transformer'):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[6, 10], gamma=0.6)
    best_acc = 0
    early_stopping = EarlyStopping(patience=20)

    train_losses = []
    val_accuracies = []

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0
        correct = 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch}"):
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids)
            loss = F.cross_entropy(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()

        train_acc = correct / len(train_loader.dataset)
        avg_loss = total_loss
        train_losses.append(avg_loss)
        print(f">>> Epoch {epoch} Train Loss: {avg_loss:.4f}, Accuracy: {train_acc:.4f}")

        # evaluate
        val_acc = evaluate_model(model, val_loader, device)
        val_accuracies.append(val_acc)

        if val_acc > best_acc:
            best_acc = val_acc
            print(">>> Best model updated. Saving...")
            torch.save(model.state_dict(), f"{name}_model.pt")

        scheduler.step()

        early_stopping(val_acc, model, path=f"{name}_model.pt")
        if early_stopping.early_stop:
            print(">>> Early stopping triggered. Stop training.")
            break

    return train_losses, val_accuracies

def evaluate_model(model, dataloader, device):
    model.eval()
    correct = 0
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids)
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
    acc = correct / len(dataloader.dataset)
    print(f">>> Validation Accuracy: {acc:.4f}")
    return acc

def test_model(model, dataloader, device):
    model.eval()
    y_true = []
    y_pred = []
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids)
            preds = torch.argmax(outputs, dim=1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())
    acc = accuracy_score(y_true, y_pred)
    print(f">>> Test Accuracy: {acc:.4f}")
    print(classification_report(y_true, y_pred, target_names=label_list, digits=3))

class EarlyStopping:
    def __init__(self, patience=3, verbose=True):
        self.patience = patience
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.verbose = verbose

    def __call__(self, score, model, path='Transformer_model.pt'):
        if self.best_score is None or score > self.best_score:
            self.best_score = score
            self.counter = 0
            torch.save(model.state_dict(), path)
            if self.verbose:
                print(f">>> New best score: {score:.4f}, saving model.")
        else:
            self.counter += 1
            if self.verbose:
                print(f">>> No improvement. EarlyStop counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True

def plot_training_curve(train_losses, val_accuracies):
    epochs = range(1, len(train_losses)+1)

    fig, ax1 = plt.subplots(figsize=(10, 5))

    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Train Loss", color='tab:red')
    ax1.plot(epochs, train_losses, label="Train Loss", color='tab:red')
    ax1.tick_params(axis='y', labelcolor='tab:red')

    ax2 = ax1.twinx()
    ax2.set_ylabel("Validation Accuracy", color='tab:blue')
    ax2.plot(epochs, val_accuracies, label="Val Accuracy", color='tab:blue')
    ax2.tick_params(axis='y', labelcolor='tab:blue')

    plt.title("Training Loss & Validation Accuracy")
    fig.tight_layout()
    plt.show()

In [ ]:
# ========= TextRCNN =========
class TextRCNN(nn.Module):
    def __init__(self,
                 vocab_size = 21128, 
                 n_class = 2,  
                 embed_dim=256,  
                 rnn_hidden=384,
                 dropout=0.1
                 ):
        super(TextRCNN, self).__init__()
        self.rnn_hidden = rnn_hidden
        self.embed_dim = embed_dim
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
        )
        self.maxpool = nn.MaxPool1d(64)
        self.lstm = nn.LSTM(embed_dim, rnn_hidden, 2,
                            bidirectional=True, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(in_features=embed_dim+2*rnn_hidden,
                            out_features=n_class)

    def forward(self, x):
        #[batch, text_len]
        x = self.embedding(x)
        # [batch, text_len, embed_dim]
        # output, h_n = self.gru(x)
        output, _ = self.lstm(x)
        x = torch.cat([x, output], dim=2)
        # [batch, text_len, 2*rnn_hidden+embed_dim]
        x = F.relu(x)
        x = x.permute(0, 2, 1)
        x = self.maxpool(x).squeeze()
        # x = F.max_pool2d(x, (x.shape[1], 1))
        # [batch, 1, 2*rnn_hidden+embed_dim]
        # x = x.reshape(-1,2 * self.rnn_hidden+self.embed_dim)
        # [batch, 2*rnn_hidden+embed_dim]
        x = self.fc(x)   # [batch, n_class]
        # x = torch.sigmoid(x)
        return x

In [ ]:
# ========= TextRNN =========
class TextRNN(nn.Module):
    def __init__(self,
                 vocab_size=21128,
                 embed_dim=256,
                 hidden_size=256,
                 n_class=2,
                 num_layers=2,
                 dropout=0.2):
        super(TextRNN, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(input_size=embed_dim,
                            hidden_size=hidden_size,
                            num_layers=num_layers,
                            batch_first=True,
                            bidirectional=True,
                            dropout=dropout)
        self.fc = nn.Linear(hidden_size * 2, n_class)

    def forward(self, input_ids):
        x = self.embedding(input_ids)  # [batch, seq_len, embed_dim]
        output, _ = self.lstm(x)       # [batch, seq_len, hidden*2]
        out = output[:, -1, :]         
        return self.fc(out)            # [batch, n_class]

In [ ]:
# ========= TextRNN_Attention =========
class TextRNN_Attention(nn.Module):
    def __init__(self,
                 vocab_size=21128,
                 embed_dim=256,
                 hidden_size=384,
                 n_class=2,
                 num_layers=3,
                 dropout=0.2):
        super(TextRNN_Attention, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim,
                            hidden_size,
                            num_layers=num_layers,
                            bidirectional=True,
                            batch_first=True,
                            dropout=dropout)

        self.tanh = nn.Tanh()
        self.attention_weight = nn.Parameter(torch.zeros(hidden_size * 2))  
        self.fc = nn.Linear(hidden_size * 2, n_class)

    def forward(self, input_ids):
        x = self.embedding(input_ids)                    # [batch, seq_len, embed_dim]
        output, _ = self.lstm(x)                         # [batch, seq_len, hidden*2]

        M = self.tanh(output)                            # [batch, seq_len, hidden*2]
        alpha = F.softmax(torch.matmul(M, self.attention_weight), dim=1).unsqueeze(-1)
                                                         # [batch, seq_len, 1]
        out = output * alpha                             # [batch, seq_len, hidden*2]
        out = torch.sum(out, dim=1)                      # [batch, hidden*2]
        out = F.relu(out)
        logits = self.fc(out)                            # [batch, num_classes]
        return logits


When switching to a different model, make sure to update the following:

1.Comment out the unused model = ... line
Only keep the instantiation for the model you want to use.

2.Update the model name inside train_model(...)
Make sure the name parameter matches the selected model (used for saving .pt files).

3.Comment out unused model.load_state_dict(...) lines
Only keep the one corresponding to the model you want to load for evaluation or prediction.

In [ ]:
# ========= main =========


#train_df = pd.read_csv(os.path.join(data_path, train_file))
val_df = pd.read_csv(os.path.join(data_path, valid_file))
test_df = pd.read_csv(os.path.join(data_path, test_file))

train_dataset = CommentDataset(
    texts=train_df['text'].tolist(),
    labels=train_df['label'].tolist(),
    word_freq=word_freq,         
    max_len=fix_length
)
val_dataset = CommentDataset(val_df['text'].tolist(), val_df['label'].tolist(), max_len=fix_length)
test_dataset = CommentDataset(test_df['text'].tolist(), test_df['label'].tolist(), max_len=fix_length)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

# model train
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Transformer()
#model = TextCNN()
#model = TextRCNN()
#model = TextRNN()
#model = TextRNN_Attention()
train_losses, val_accuracies = train_model(model, train_loader, val_loader, device, name='Transformer')
plot_training_curve(train_losses, val_accuracies)
# model test
print(">>> Loading best model and testing...")
model.load_state_dict(torch.load("Transformer_model.pt"))
#model.load_state_dict(torch.load("TextCNN_model.pt"))
#model.load_state_dict(torch.load("TextRCNN_model.pt"))
#model.load_state_dict(torch.load("TextRNN_model.pt"))
#model.load_state_dict(torch.load("TextRNN_Attention_model.pt"))
test_model(model, test_loader, device)